# UKRI FoR Classifier — POC Batch Inference

This notebook is a **POC deployment/inference workflow** for the FoR multi-label classifier in Ronin.

### Pipeline
1. Connect to S3 with `boto3`
2. Download the staging input file to the local Unix drive
3. Validate the input schema
4. Build a unique application key from `ApplicationID + ApplicationOriginSource`
5. Remove records where both `ApplicationTitle` and `ApplicationSummary` are null/blank
6. Build model text as `ApplicationTitle + " " + ApplicationSummary`
7. Clean and lemmatise text
8. Load:
   - `tfidf_logreg.pkl`
   - `tfidf_thresholds.pkl`
   - `mlb_for_taxonomy_groups.pkl`
9. Run `predict_proba`
10. Apply saved per-class thresholds
11. Produce one output row per predicted FoR category
12. Validate the output
13. Save a local copy
14. Upload the output to S3

> **Important:** Do not place AWS access keys in this notebook. `boto3` should use the IAM credentials/role already provided by Ronin.

## 0. Configuration

Update the S3 bucket, input key/prefix and output prefix before running.

Two input modes are supported:

- **Specific key:** set `S3_INPUT_KEY`
- **Latest file under prefix:** set `S3_INPUT_KEY = None` and configure `S3_INPUT_PREFIX`

The output filename follows:

`yyyymmddhhmm_Taxonomy_ModelName_ModelVersion`

Example: `202607310327_FoR_FoRClassification_1.1.csv`

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import re
import html
import json
import unicodedata

# ----------------------------
# Project paths
# ----------------------------
CURRENT_DIR = Path.cwd()

# Works whether the notebook is run from project root or /notebooks
if (CURRENT_DIR / "models").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "models").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR.parent

DATA_DIR = PROJECT_ROOT / "data"
INPUT_DIR = DATA_DIR / "input"
OUTPUT_DIR = DATA_DIR / "output"
REJECTED_DIR = DATA_DIR / "rejected"
LOG_DIR = PROJECT_ROOT / "logs"
MODEL_DIR = PROJECT_ROOT / "models"

for directory in [INPUT_DIR, OUTPUT_DIR, REJECTED_DIR, LOG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ----------------------------
# S3 configuration - UPDATE
# ----------------------------
S3_BUCKET = "<YOUR-STAGING-BUCKET>"

# Option A: exact input object key
S3_INPUT_KEY = None
# Example:
# S3_INPUT_KEY = "some/prefix/input_file.csv"

# Option B: automatically select the newest object under this prefix
S3_INPUT_PREFIX = "<YOUR-INPUT-PREFIX>/"

S3_OUTPUT_PREFIX = "<YOUR-OUTPUT-PREFIX>/"

# Optional suffix filter when selecting latest object
INPUT_SUFFIXES = (".csv", ".parquet")

# ----------------------------
# Model release configuration
# ----------------------------
MODEL_PATH = MODEL_DIR / "tfidf_logreg.pkl"
THRESHOLDS_PATH = MODEL_DIR / "tfidf_thresholds.pkl"
MLB_PATH = MODEL_DIR / "mlb_for_taxonomy_groups.pkl"

TAXONOMY_FILE_TOKEN = "FoR"
TAXONOMY_VALUE = "FieldsOfResearch"
MODEL_NAME = "FoRClassification"
MODEL_VERSION = "1.1"
SCORE_TYPE = "uncalibrated"

# Set to .csv or .parquet once agreed with stakeholders.
OUTPUT_EXTENSION = ".csv"

# Training notebook showed a clean_text() stage before lemmatisation.
# Human-evaluation notebook lemmatised full_text directly.
# Keep True to mirror the training pipeline; set False only if the released
# model artefact was explicitly validated without this step.
APPLY_CLEAN_TEXT = True

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODEL_DIR:", MODEL_DIR)
print("INPUT_DIR:", INPUT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 1. Imports and environment checks

The production Docker image should contain the required spaCy model.  
This notebook **does not download it from the internet at runtime**.

In [ ]:
import sys
import boto3
import botocore
import joblib
import numpy as np
import pandas as pd
import sklearn
import spacy

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("spaCy:", spacy.__version__)
print("boto3:", boto3.__version__)

# Confirm the three required model artefacts exist.
for p in [MODEL_PATH, THRESHOLDS_PATH, MLB_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required model artefact not found: {p}")

# Load the approved spaCy language model.
# Do not dynamically download it in production.
try:
    nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])
    print("Loaded spaCy model: en_core_web_sm")
except OSError as exc:
    raise RuntimeError(
        "spaCy model 'en_core_web_sm' is not installed in this environment. "
        "Install/package the approved version before running inference."
    ) from exc

## 2. Test AWS / S3 connectivity

This verifies that `boto3` can use the Ronin AWS credentials/role.

If the existing `aws s3 cp` command works without manually passing credentials, `boto3` will normally use the same AWS credential chain.

In [ ]:
session = boto3.Session()
s3 = session.client("s3")
sts = session.client("sts")

identity = sts.get_caller_identity()

print("AWS Account:", identity.get("Account"))
print("Caller ARN:", identity.get("Arn"))

# Lightweight access check.
# This does not list every bucket; it checks the configured bucket directly.
try:
    s3.head_bucket(Bucket=S3_BUCKET)
    print(f"Successfully accessed bucket: s3://{S3_BUCKET}")
except botocore.exceptions.ClientError as exc:
    raise RuntimeError(
        f"Unable to access configured S3 bucket: s3://{S3_BUCKET}. "
        "Check bucket name, IAM permissions and Ronin networking."
    ) from exc

## 3. Select the S3 input object

If `S3_INPUT_KEY` is not supplied, the notebook chooses the **most recently modified** CSV/Parquet object under `S3_INPUT_PREFIX`.

For production, the final Airflow pipeline may instead receive the exact S3 key as a run parameter.

In [ ]:
def find_latest_s3_object(
    s3_client,
    bucket: str,
    prefix: str,
    allowed_suffixes=(".csv", ".parquet"),
) -> str:
    paginator = s3_client.get_paginator("list_objects_v2")
    candidates = []

    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.lower().endswith(tuple(s.lower() for s in allowed_suffixes)):
                candidates.append(obj)

    if not candidates:
        raise FileNotFoundError(
            f"No matching files found under s3://{bucket}/{prefix}"
        )

    latest = max(candidates, key=lambda x: x["LastModified"])
    return latest["Key"]

if S3_INPUT_KEY:
    selected_input_key = S3_INPUT_KEY
else:
    selected_input_key = find_latest_s3_object(
        s3,
        S3_BUCKET,
        S3_INPUT_PREFIX,
        INPUT_SUFFIXES,
    )

print("Selected S3 input:", f"s3://{S3_BUCKET}/{selected_input_key}")

## 4. Download input from S3 to the local Unix drive

In [ ]:
local_input_path = INPUT_DIR / Path(selected_input_key).name

s3.download_file(
    S3_BUCKET,
    selected_input_key,
    str(local_input_path),
)

if not local_input_path.exists():
    raise FileNotFoundError(f"Download failed: {local_input_path}")

if local_input_path.stat().st_size == 0:
    raise ValueError(f"Downloaded input is empty: {local_input_path}")

print("Downloaded to:", local_input_path)
print("Size (bytes):", local_input_path.stat().st_size)

## 5. Read input file

CSV and Parquet are supported in this POC.

In [ ]:
def read_input_file(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(path)
    elif suffix == ".parquet":
        return pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported input file format: {suffix}")

df_raw = read_input_file(local_input_path)

print("Rows:", len(df_raw))
print("Columns:", len(df_raw.columns))
display(df_raw.head(3))

## 6. Validate input schema and unique application identity

Stakeholder-confirmed rules:

- `ApplicationID` alone is **not** a unique identifier.
- Application-level uniqueness is:
  `ApplicationID + ApplicationOriginSource`
- `LeadFundingArea` and `Administrator` are not used by the model.
- `ApplicationTitle` and `ApplicationSummary` are the only text fields used for inference.

In [ ]:
REQUIRED_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "ApplicationTitle",
    "ApplicationSummary",
]

missing_columns = [c for c in REQUIRED_COLUMNS if c not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Input missing required columns: {missing_columns}")

if len(df_raw) == 0:
    raise ValueError("Input contains zero rows.")

# Work on a copy and keep stakeholder identifiers as strings.
df = df_raw.copy()

df["ApplicationID"] = df["ApplicationID"].astype("string")
df["ApplicationOriginSource"] = df["ApplicationOriginSource"].astype("string")

# ApplicationOriginSource is essential to identity/output traceability.
if df["ApplicationOriginSource"].isna().any():
    raise ValueError(
        "ApplicationOriginSource contains null values. "
        "It is required to build the composite application identity."
    )

# Composite key kept internally; not required in final output.
df["_application_key"] = (
    df["ApplicationID"].fillna("<NULL>")
    + "||"
    + df["ApplicationOriginSource"].fillna("<NULL>")
)

duplicate_app_keys = df["_application_key"].duplicated(keep=False)

print("Input rows:", len(df))
print("Duplicate ApplicationID + ApplicationOriginSource rows:",
      int(duplicate_app_keys.sum()))

# Do not fail here automatically because a source file may contain duplicate
# rows that stakeholders want investigated. Review before productionising.
if duplicate_app_keys.any():
    display(
        df.loc[
            duplicate_app_keys,
            ["ApplicationID", "ApplicationOriginSource"]
        ].head(20)
    )

## 7. Null/blank title-summary handling

Rules:

- Title present + Summary present → use both
- Title present + Summary null → use title
- Title null + Summary present → use summary
- Both null/blank → do not send to the model

Rows with no model text are retained separately with rejection reason `NO_MODEL_TEXT`.

In [ ]:
def normalise_nullable_text(series: pd.Series) -> pd.Series:
    # Preserve true missingness long enough to identify empty model records.
    s = series.astype("string")
    s = s.str.strip()
    s = s.replace("", pd.NA)
    return s

df["ApplicationTitle"] = normalise_nullable_text(df["ApplicationTitle"])
df["ApplicationSummary"] = normalise_nullable_text(df["ApplicationSummary"])

no_model_text_mask = (
    df["ApplicationTitle"].isna()
    & df["ApplicationSummary"].isna()
)

df_rejected = df.loc[no_model_text_mask].copy()
df_rejected["rejection_reason"] = "NO_MODEL_TEXT"

df_model = df.loc[~no_model_text_mask].copy()

# Exact stakeholder-confirmed concatenation format:
# ApplicationTitle + " " + ApplicationSummary
df_model["full_text"] = (
    df_model["ApplicationTitle"].fillna("")
    + " "
    + df_model["ApplicationSummary"].fillna("")
).str.strip()

print("Total input rows:", len(df))
print("Rows sent to model:", len(df_model))
print("Rows rejected (both title and summary empty):", len(df_rejected))

if len(df_model) == 0:
    raise ValueError("No records contain usable model text.")

## 8. Text cleaning and lemmatisation

The training notebook showed:

- Unicode NFKC normalisation
- HTML unescape
- whitespace normalisation
- spaCy lemmatisation
- removal of stopwords
- removal of punctuation
- removal of tokens with length <= 2

The saved `tfidf_logreg.pkl` already contains the fitted TF-IDF vectorizer, so **do not fit a new TF-IDF vectorizer during inference**.

In [ ]:
def clean_text(text):
    if pd.isnull(text):
        return ""

    text = str(text)
    text = unicodedata.normalize("NFKC", text)
    text = html.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def lemmatize_text_noisalpha(texts, n_process=None, batch_size=50):
    """
    Mirrors the training/human-evaluation preprocessing shown in the notebooks.
    Despite the historical function name, the active training code shown
    filters stopwords, punctuation and tokens <= 2 characters.
    """
    if n_process is None:
        # Conservative default for a shared POC box.
        n_process = max(1, (os.cpu_count() or 2) - 1)

    cleaned_texts = []

    for doc in nlp.pipe(
        texts,
        n_process=n_process,
        batch_size=batch_size,
    ):
        lemma_text = " ".join(
            token.lemma_
            for token in doc
            if not token.is_stop
            and not token.is_punct
            and len(token.text) > 2
        )
        cleaned_texts.append(lemma_text)

    return cleaned_texts


if APPLY_CLEAN_TEXT:
    df_model["clean_text"] = df_model["full_text"].apply(clean_text)
    text_for_lemma = df_model["clean_text"]
else:
    df_model["clean_text"] = df_model["full_text"]
    text_for_lemma = df_model["full_text"]

df_model["lemmatized_full_text"] = lemmatize_text_noisalpha(
    text_for_lemma.tolist()
)

if (df_model["lemmatized_full_text"].str.strip() == "").any():
    print(
        "WARNING:",
        int((df_model["lemmatized_full_text"].str.strip() == "").sum()),
        "records became empty after preprocessing."
    )

display(
    df_model[
        [
            "ApplicationID",
            "ApplicationOriginSource",
            "full_text",
            "lemmatized_full_text",
        ]
    ].head(3)
)

## 9. Load released model artefacts

These three files form one logical model release:

- `tfidf_logreg.pkl`
- `tfidf_thresholds.pkl`
- `mlb_for_taxonomy_groups.pkl`

In [ ]:
model = joblib.load(MODEL_PATH)
thresholds = np.asarray(joblib.load(THRESHOLDS_PATH))
mlb = joblib.load(MLB_PATH)

print("Model type:", type(model))
print("Threshold shape:", thresholds.shape)
print("MLB classes:", len(mlb.classes_))

# Show pipeline steps if this is the expected sklearn Pipeline.
if hasattr(model, "named_steps"):
    print("Pipeline steps:", list(model.named_steps.keys()))

## 10. Model artefact compatibility checks

The screenshots showed 171 model output classes.  
Do not hard-code 171 for future releases; instead validate the artefacts against one another.

In [ ]:
# Flatten thresholds in the same way as the human-evaluation notebook.
thresholds = thresholds.reshape(-1)

n_thresholds = len(thresholds)
n_mlb_classes = len(mlb.classes_)

if n_thresholds != n_mlb_classes:
    raise ValueError(
        "Model release mismatch: "
        f"{n_thresholds} thresholds vs {n_mlb_classes} MLB classes."
    )

# If OneVsRestClassifier is accessible inside the saved pipeline,
# compare the number of fitted estimators as an extra safeguard.
n_model_classes = None

if hasattr(model, "named_steps") and "clf" in model.named_steps:
    clf = model.named_steps["clf"]

    if hasattr(clf, "estimators_"):
        n_model_classes = len(clf.estimators_)
    elif hasattr(clf, "classes_"):
        try:
            n_model_classes = len(clf.classes_)
        except TypeError:
            pass

if n_model_classes is not None and n_model_classes != n_thresholds:
    raise ValueError(
        "Model release mismatch: "
        f"{n_model_classes} model outputs vs {n_thresholds} thresholds."
    )

print("Artefact compatibility check passed.")
print("Output classes:", n_thresholds)

## 11. Run inference

The saved sklearn pipeline performs:

`lemmatised text → fitted TF-IDF → OneVsRest logistic regression → probabilities`

In [ ]:
X_inference = df_model["lemmatized_full_text"]

y_probs = model.predict_proba(X_inference)
y_probs = np.asarray(y_probs)

print("Probability matrix shape:", y_probs.shape)

if y_probs.ndim != 2:
    raise ValueError(
        f"Expected 2-D probability matrix, got shape {y_probs.shape}"
    )

if y_probs.shape[0] != len(df_model):
    raise ValueError(
        "Prediction row count does not match model input row count."
    )

if y_probs.shape[1] != n_thresholds:
    raise ValueError(
        "Probability class count does not match saved thresholds: "
        f"{y_probs.shape[1]} vs {n_thresholds}"
    )

## 12. Apply the saved per-class thresholds

Do **not** retune thresholds in production inference.

The released thresholds are applied class-by-class:

`predicted = probability >= saved_threshold`

In [ ]:
y_pred = (y_probs >= thresholds.reshape(1, -1)).astype(int)

predictions_per_application = y_pred.sum(axis=1)

print("Applications scored:", len(y_pred))
print("Applications with zero predicted categories:",
      int((predictions_per_application == 0).sum()))
print("Applications with one predicted category:",
      int((predictions_per_application == 1).sum()))
print("Applications with multiple predicted categories:",
      int((predictions_per_application > 1).sum()))
print("Maximum predicted categories for one application:",
      int(predictions_per_application.max()))

## 13. Decode FoR categories and create long-format prediction rows

One application can have one or many FoR categories.

The final output therefore contains one row per:

`ApplicationID + ApplicationOriginSource + category_id`

The `score` is the model probability for that specific selected category.

In [ ]:
prediction_rows = []

# mlb.classes_[j] maps probability column j to the FoR class/category.
for row_pos, (_, source_row) in enumerate(df_model.iterrows()):
    selected_indices = np.flatnonzero(y_pred[row_pos] == 1)

    for class_idx in selected_indices:
        category = mlb.classes_[class_idx]
        score = float(y_probs[row_pos, class_idx])

        prediction_rows.append(
            {
                "ApplicationID": source_row["ApplicationID"],
                "ApplicationOriginSource": source_row["ApplicationOriginSource"],
                "category_id": category,
                "score": score,
            }
        )

df_predictions = pd.DataFrame(
    prediction_rows,
    columns=[
        "ApplicationID",
        "ApplicationOriginSource",
        "category_id",
        "score",
    ],
)

print("Prediction output rows:", len(df_predictions))
display(df_predictions.head(10))

## 14. Build the stakeholder-required output schema

Rules:

- `model_run_date` = system date when inference runs
- `score_type` = fixed `uncalibrated`
- `taxonomy` = fixed `FieldsOfResearch`
- `ApplicationOriginSource` remains unchanged from input
- `score` is rounded to 2 decimal places to align with `Decimal(5,2)`

In [ ]:
run_timestamp = datetime.now()
model_run_date = run_timestamp.date()

df_output = df_predictions.copy()

df_output["model_run_date"] = model_run_date
df_output["score_type"] = SCORE_TYPE
df_output["score"] = pd.to_numeric(df_output["score"], errors="raise").round(2)
df_output["taxonomy"] = TAXONOMY_VALUE

# Best-effort conversion to integer category ID while failing loudly
# if the released MLB classes cannot represent integer FoR category IDs.
category_numeric = pd.to_numeric(df_output["category_id"], errors="coerce")

if len(df_output) > 0 and category_numeric.isna().any():
    bad_values = df_output.loc[
        category_numeric.isna(), "category_id"
    ].astype(str).unique()[:10]

    raise ValueError(
        "Some decoded category IDs are not numeric. Examples: "
        f"{bad_values.tolist()}"
    )

if len(df_output) > 0:
    # Handles values such as 3101 or 3101.0.
    if not np.allclose(category_numeric, np.round(category_numeric)):
        raise ValueError(
            "Some category IDs are numeric but not whole numbers."
        )

    df_output["category_id"] = np.round(category_numeric).astype("int64")
else:
    df_output["category_id"] = pd.Series(dtype="int64")

FINAL_COLUMNS = [
    "ApplicationID",
    "ApplicationOriginSource",
    "model_run_date",
    "category_id",
    "score_type",
    "score",
    "taxonomy",
]

df_output = df_output[FINAL_COLUMNS]

display(df_output.head(10))

## 15. Validate final output

Output-level uniqueness is:

`ApplicationID + ApplicationOriginSource + category_id`

In [ ]:
OUTPUT_KEY = [
    "ApplicationID",
    "ApplicationOriginSource",
    "category_id",
]

# Required fields must be present.
missing_output_columns = [
    c for c in FINAL_COLUMNS if c not in df_output.columns
]

if missing_output_columns:
    raise ValueError(
        f"Final output missing columns: {missing_output_columns}"
    )

duplicate_output_key = df_output.duplicated(
    subset=OUTPUT_KEY,
    keep=False,
)

if duplicate_output_key.any():
    display(df_output.loc[duplicate_output_key].head(20))
    raise ValueError(
        "Duplicate output key detected for "
        "ApplicationID + ApplicationOriginSource + category_id."
    )

if df_output["score"].notna().any():
    if ((df_output["score"] < 0) | (df_output["score"] > 1)).any():
        raise ValueError("Model scores must be between 0 and 1.")

if not (df_output["score_type"] == SCORE_TYPE).all():
    raise ValueError("Unexpected score_type value.")

if not (df_output["taxonomy"] == TAXONOMY_VALUE).all():
    raise ValueError("Unexpected taxonomy value.")

# Ensure every output row traces back to an input application identity.
input_keys = set(
    zip(
        df["ApplicationID"].astype(str),
        df["ApplicationOriginSource"].astype(str),
    )
)

output_keys = set(
    zip(
        df_output["ApplicationID"].astype(str),
        df_output["ApplicationOriginSource"].astype(str),
    )
)

orphan_keys = output_keys - input_keys

if orphan_keys:
    raise ValueError(
        f"Output contains application identities not found in input: "
        f"{list(orphan_keys)[:10]}"
    )

print("Final output validation passed.")
print("Output rows:", len(df_output))
print("Unique source applications with predictions:",
      df_output[["ApplicationID", "ApplicationOriginSource"]]
      .drop_duplicates()
      .shape[0])

## 16. Generate the agreed output filename

Format:

`yyyymmddhhmm_Taxonomy_ModelName_ModelVersion`

The run metadata also records:

- `model_name`
- `model_version`
- `data_creator = Automated Process`

In [ ]:
filename_timestamp = run_timestamp.strftime("%Y%m%d%H%M")

output_filename = (
    f"{filename_timestamp}_"
    f"{TAXONOMY_FILE_TOKEN}_"
    f"{MODEL_NAME}_"
    f"{MODEL_VERSION}"
    f"{OUTPUT_EXTENSION}"
)

local_output_path = OUTPUT_DIR / output_filename

print("Output filename:", output_filename)
print("Local path:", local_output_path)

## 17. Save final output locally

In [ ]:
def save_output_file(df_out: pd.DataFrame, path: Path) -> None:
    suffix = path.suffix.lower()

    if suffix == ".csv":
        df_out.to_csv(path, index=False)
    elif suffix == ".parquet":
        df_out.to_parquet(path, index=False)
    else:
        raise ValueError(f"Unsupported output format: {suffix}")

save_output_file(df_output, local_output_path)

if not local_output_path.exists():
    raise FileNotFoundError(
        f"Local output was not created: {local_output_path}"
    )

print("Saved local output:", local_output_path)
print("Output size (bytes):", local_output_path.stat().st_size)

## 18. Save skipped/no-text records locally

This file is for POC traceability and is **not** part of the agreed model output contract unless stakeholders request it.

In [ ]:
rejected_filename = (
    f"{filename_timestamp}_"
    f"{TAXONOMY_FILE_TOKEN}_"
    f"{MODEL_NAME}_"
    f"{MODEL_VERSION}_rejected.csv"
)

local_rejected_path = REJECTED_DIR / rejected_filename

rejected_columns = [
    "ApplicationID",
    "ApplicationOriginSource",
    "ApplicationTitle",
    "ApplicationSummary",
    "rejection_reason",
]

df_rejected[rejected_columns].to_csv(
    local_rejected_path,
    index=False,
)

print("Rejected rows:", len(df_rejected))
print("Rejected file:", local_rejected_path)

## 19. Upload final output to S3

The final S3 upload is deliberately performed **only after local output validation succeeds**.

In [ ]:
s3_output_key = (
    S3_OUTPUT_PREFIX.rstrip("/")
    + "/"
    + output_filename
)

s3.upload_file(
    str(local_output_path),
    S3_BUCKET,
    s3_output_key,
)

# Verify the object is now present.
uploaded = s3.head_object(
    Bucket=S3_BUCKET,
    Key=s3_output_key,
)

print("Uploaded:", f"s3://{S3_BUCKET}/{s3_output_key}")
print("Uploaded size (bytes):", uploaded["ContentLength"])

## 20. Create a POC run summary

This can later be sent to MLflow / Prometheus / Airflow rather than remaining as a local JSON file.

In [ ]:
run_summary = {
    "run_timestamp": run_timestamp.isoformat(),
    "model_run_date": str(model_run_date),
    "input_s3_uri": f"s3://{S3_BUCKET}/{selected_input_key}",
    "local_input_path": str(local_input_path),
    "input_rows": int(len(df)),
    "rows_sent_to_model": int(len(df_model)),
    "rows_rejected_no_model_text": int(len(df_rejected)),
    "prediction_output_rows": int(len(df_output)),
    "source_applications_with_predictions": int(
        df_output[
            ["ApplicationID", "ApplicationOriginSource"]
        ].drop_duplicates().shape[0]
    ),
    "applications_with_zero_predicted_categories": int(
        (predictions_per_application == 0).sum()
    ),
    "applications_with_multiple_predicted_categories": int(
        (predictions_per_application > 1).sum()
    ),
    "model_name": MODEL_NAME,
    "model_version": MODEL_VERSION,
    "taxonomy": TAXONOMY_VALUE,
    "score_type": SCORE_TYPE,
    "data_creator": "Automated Process",
    "model_artifact": MODEL_PATH.name,
    "threshold_artifact": THRESHOLDS_PATH.name,
    "mlb_artifact": MLB_PATH.name,
    "output_filename": output_filename,
    "local_output_path": str(local_output_path),
    "output_s3_uri": f"s3://{S3_BUCKET}/{s3_output_key}",
    "status": "SUCCESS",
}

run_summary_path = LOG_DIR / (
    f"{filename_timestamp}_{MODEL_NAME}_{MODEL_VERSION}_run_summary.json"
)

with open(run_summary_path, "w", encoding="utf-8") as f:
    json.dump(run_summary, f, indent=2)

print(json.dumps(run_summary, indent=2))
print("Run summary saved to:", run_summary_path)

## 21. Optional POC checks against a stakeholder sample output

If you place the known sample output under `data/output/expected/`, compare:

- column names
- data types
- uniqueness
- application/source/category combinations
- score values

Do not enable this as a production dependency; it is intended as a POC regression test.

In [ ]:
# Example only — uncomment and update the path when required.
#
# EXPECTED_OUTPUT_PATH = OUTPUT_DIR / "expected" / "sample_output.csv"
#
# if EXPECTED_OUTPUT_PATH.exists():
#     df_expected = pd.read_csv(EXPECTED_OUTPUT_PATH)
#
#     print("Expected columns:", df_expected.columns.tolist())
#     print("Actual columns:  ", df_output.columns.tolist())
#     print("Expected rows:", len(df_expected))
#     print("Actual rows:", len(df_output))
#
#     assert df_expected.columns.tolist() == df_output.columns.tolist()
# else:
#     print("No expected sample output configured.")

# Next step after the notebook works

Once **Restart Kernel → Run All** succeeds reliably:

1. Move S3 functions to `src/s3_io.py`
2. Move validation to `src/validation.py`
3. Move preprocessing to `src/preprocessing.py`
4. Move model loading/inference to `src/inference.py`
5. Move output construction to `src/output_builder.py`
6. Create a single `run_pipeline()` entry point
7. Add tests using the supplied sample input/output
8. Package in Docker
9. Let Airflow call the Dockerised pipeline

This keeps the POC notebook as a reference implementation while the deployment code becomes modular and testable.